# Module: Analyzing Business Problems
## Sprint: Marketing Analyst
## Part 2: Hands-on exercise

## Your role
You are a marketing analyst in a company specializing in an omnichannel marketing automation platform for ecommerce business with email marketing at the core. As it is a relatively small startup and you are known in the company to be “good with data” you get to analyze all kinds of marketing related data, and sometimes even data not directly related to marketing. Your main stakeholder is the marketing manager, who is responsible for running all marketing activities in the company, though you get requests from product managers, researchers and others.

## Data
You will have multiple datasets which will help you answer the main questions. One of the first datasets is a table called [`turing_data_analytics.stackoverflow_post`](https://console.cloud.google.com/bigquery?authuser=3&project=tc-da-1&ws=!1m9!1m3!8m2!1s756497109418!2s1b8fa7cb3d73401f81fea1591fa615a7!1m4!4m3!1stc-da-1!2sturing_data_analytics!3sstackoverflow_posts&d=turing_data_analytics&p=tc-da-1&t=stackoverflow_posts&page=table) containing data from 2016-01-01 till 2016-09-21. You can see columns in the table detail section. This dataset contains data from Stack Overflow website, which acts as a question & answer website, mostly around programming questions. This dataset contains all questions asked on the platform no matter the topic.

Another dataset called [`turing_data_analytics.adsense_monthly`](https://console.cloud.google.com/bigquery?authuser=3&project=tc-da-1&ws=!1m19!1m3!8m2!1s756497109418!2s1b8fa7cb3d73401f81fea1591fa615a7!1m4!4m3!1stc-da-1!2sturing_data_analytics!3sstackoverflow_posts!1m4!4m3!1stc-da-1!2sturing_data_analytics!3sraw_events!1m4!4m3!1stc-da-1!2sturing_data_analytics!3sadsense_monthly&d=turing_data_analytics&p=tc-da-1&t=adsense_monthly&page=table) contains monthly data for marketing spending for a period from 2020-11 till 2021-02 for the company you are working for. You have a single parsed events table which contains various frontend actions done on your site. Table is called [`turing_data_analytics.raw_events`](https://console.cloud.google.com/bigquery?authuser=3&project=tc-da-1&ws=!1m14!1m3!8m2!1s756497109418!2s1b8fa7cb3d73401f81fea1591fa615a7!1m4!4m3!1stc-da-1!2sturing_data_analytics!3sstackoverflow_posts!1m4!4m3!1stc-da-1!2sturing_data_analytics!3sraw_events&d=turing_data_analytics&p=tc-da-1&t=raw_events&page=table) and columns description are available in the table details section. Data in the table contains records from 2020-11-01 until 2021-01-31.

## Monday
Your marketing manager comes over and lets you know that the product team is thinking of having an integration with WooCommerce – an e-commerce platform that they seem to be hearing quite a lot about recently. They are interested if there is enough engagement and support for WooCommerce in public question and answer websites.  They choose to analyze Stack Overflow, as it is the most popular website for both professional and enthusiast programmers.

Your product manager asked a developer to [scrape information from Stack Overflow](https://console.cloud.google.com/bigquery?authuser=3&project=tc-da-1&ws=!1m9!1m3!8m2!1s756497109418!2s1b8fa7cb3d73401f81fea1591fa615a7!1m4!4m3!1stc-da-1!2sturing_data_analytics!3sstackoverflow_posts&d=turing_data_analytics&p=tc-da-1&t=stackoverflow_posts&page=table) website that could help you answer the question of WooCommerce support. The data scraped contains different dimensions and metrics: tags, answer count, comment count, description, favorite count, last activity dates, score, view count etc. The data sample contains data for all posts on Stack Overflow – no topics were excluded.


You help your marketing manager to come up with an engagement rate definition and define it as favorite ratio (favorite count / view count). **Write a query which would show the monthly favorite ratio for all posts on Stack Overflow in this period.**



In [ ]:
SELECT
FORMAT_DATE("%B",DATE_TRUNC(creation_date, MONTH)) AS month,
EXTRACT(MONTH FROM creation_date) AS month_num,
SUM(favorite_count)/SUM(view_count) AS favorite_ratio
FROM
`tc-da-1.turing_data_analytics.stackoverflow_posts` stack
WHERE
stack.creation_date >= "2016-01-01"
GROUP BY 1,2
ORDER BY month_num ASC;

# other answers may be acceptable as long as results are same


Based on this SQL query you [visualize your data using a bar chart](https://docs.google.com/spreadsheets/d/1DKh2zCDuuaHqkm7fvWsjSFVqF1R5CJvPhyJ27Ibo3_E/edit#gid=1826538439).

Your marketing manager thanks you and asks to add some splits to this data. She wants to see if this dynamic applies to Woocommerce questions as well and how they benchmark against the rest of the questions. You decide to modify your query to provide splits by tag types. For this you use the `tags` field by checking if it contains the `woocommerce` keyword, the ones not containing it you group into `other` category.

**Write a query to achieve this.**


In [ ]:
SELECT
EXTRACT(MONTH FROM creation_date) AS month_num,
CASE WHEN CONTAINS_SUBSTR(tags, "woocommerce") THEN "woocommerce" ELSE "other" END AS tag_group,
FORMAT_DATE("%B",DATE_TRUNC(creation_date, MONTH)) AS month,
SUM(favorite_count)/SUM(view_count) AS favorite_ratio
FROM
`tc-da-1.turing_data_analytics.stackoverflow_posts` stack
WHERE
stack.creation_date >= "2016-01-01"
GROUP BY 1,2,3
ORDER BY month_num, tag_group ASC;

Based on this query you [visualize your data](https://docs.google.com/spreadsheets/d/1DKh2zCDuuaHqkm7fvWsjSFVqF1R5CJvPhyJ27Ibo3_E/edit#gid=1826538439).

You explain to your marketing manager that the overall favorite ratio is increasing over time for Stack Overflow, this indicates that users are getting more and more engaged with the platform. Though it’s clear that Woocommerce usually trends lower than other tags, except for the month of April. 

Based on all of this, your manager asks to evaluate the quality of answers using the Stack Overflow’s score metric for both categories and show the average of it over 6 months, again splitting "woocommerce" and "other" categories. Additionally, he asks to filter only answered questions.



In [ ]:
SELECT
EXTRACT(MONTH FROM creation_date) AS month_num,
CASE WHEN CONTAINS_SUBSTR(tags, "woocommerce") THEN "woocommerce" ELSE "other" END AS tag_group,
FORMAT_DATE("%B",DATE_TRUNC(creation_date, MONTH)) AS month,
AVG(score) AS avg_score
FROM
`tc-da-1.turing_data_analytics.stackoverflow_posts` stack
WHERE
stack.creation_date >= "2016-01-01"
AND stack.accepted_answer_id IS NOT NULL
GROUP BY 1,2,3
ORDER BY month_num, tag_group ASC;


And once again you present your findings in [a chart](https://docs.google.com/spreadsheets/d/1vCiRTGIitvTJGFmE5soPOT6jP_JYHWFnbVWlBnLyFq8/edit#gid=0).

You point out to your marketing manager that WooCommerce seems to perform worse than benchmark (other tags) up until May. Recently there seems to be improvement in average scores for WooCommerce related questions. Though you explain that tomorrow you will come up with a better way to calculate whether it is worth proceeding with WooCommerce integration. For this you will calculate more metrics and will even perform some statistical tests to make sure your insights are correct.

## Tuesday

Tuesday morning your marketing manager follows up with you on your promise that you will find a better way to estimate how well Woocommerce support actually works. You tell him that similarly as with marketing campaigns you could look into conversion rate (COV%). Though differently than with marketing campaigns defining conversion is slightly more tricky, as there are no actual purchases, though some questions can be treated as converted while some as not from support perspective.

**Come up with your definition of conversion and write a query which will help you pull necessary data to calculate monthly conversion rate for both groups (“WooCommerce” vs “Other”).**



In [ ]:
SELECT
EXTRACT(MONTH FROM creation_date) AS month_num,
FORMAT_DATE("%B",DATE_TRUNC(creation_date, MONTH)) AS month,
SUM(CASE WHEN CONTAINS_SUBSTR(tags, "woocommerce")=TRUE AND accepted_answer_id IS NOT NULL THEN 1 ELSE 0 END) AS woocommerce_converted_questions,
SUM(CASE WHEN CONTAINS_SUBSTR(tags, "woocommerce")=TRUE THEN 1 ELSE 0 END) AS woocommerce_questions,
SUM(CASE WHEN CONTAINS_SUBSTR(tags, "woocommerce")=FALSE AND accepted_answer_id IS NOT NULL THEN 1 ELSE 0 END) AS other_converted_questions,
SUM(CASE WHEN CONTAINS_SUBSTR(tags, "woocommerce")=FALSE THEN 1 ELSE 0 END) AS other_questions,
FROM
`tc-da-1.turing_data_analytics.stackoverflow_posts` stack
WHERE stack.creation_date >= "2016-01-01"
GROUP BY 1,2
ORDER BY month_num ASC;

You then present your findings in a [graph](https://docs.google.com/spreadsheets/d/1DKh2zCDuuaHqkm7fvWsjSFVqF1R5CJvPhyJ27Ibo3_E/edit#gid=1865988394) showing monthly conversion rates for each group.

From your graph you can clearly see that previous insights about engagement of WooCommerce questions or Stack Overflow defined score metric does not actually show the full picture. You now clearly see that Woocommerce has higher conversion rates, which indicate that these types of questions are being answered more frequently. This indicates that support for WooCommerce on Stack Overflow is actually of good quality.

You present these findings to the Product Manager arguing that WooCommerce questions on Stack Overflow get answered more often, compared to the average of other questions. Actually, the improvement is over 30%. This indicates that it would be OK to go with WooCommerce integration from Stack Overflow perspective. The Product Manager, having already some experience with some product analysts' findings for various questions, knows that improvement is not always statistically significant and asks you to check it from a statistical point of view.

For this you use already familiar tool: [A/B testing significance calculator](https://www.surveymonkey.com/mp/ab-testing-significance-calculator/) and calculate if overall conversion for all 6 months for Woocommerce is significantly higher than other questions:
<div><img src="https://i.imgur.com/KKD4fSK.png"/></div>

The calculator indicates that this is actually significant and your hypothesis that WooCommerce related questions get answered more often than other questions is correct. You present these findings to the Product Manager.

## Wednesday

Today your manager comes with some past data on marketing campaigns and asks you to calculate CPM (cost per 1000 impressions) to check their performance. You have a look at [data provided](https://console.cloud.google.com/bigquery?authuser=3&project=tc-da-1&ws=!1m19!1m3!8m2!1s756497109418!2s1b8fa7cb3d73401f81fea1591fa615a7!1m4!4m3!1stc-da-1!2sturing_data_analytics!3sstackoverflow_posts!1m4!4m3!1stc-da-1!2sturing_data_analytics!3sraw_events!1m4!4m3!1stc-da-1!2sturing_data_analytics!3sadsense_monthly&d=turing_data_analytics&p=tc-da-1&t=adsense_monthly&page=table) to you and tell him that you will include 2 more derived marketing related metrics in order to evaluate all campaigns. Have in mind that the marketing manager wants those campaign metrics on a campaign level.

Having a first look at the data you decide to split the BlackFriday_V1 campaign into 2 different ones, because you notice that this campaign was running for 2 separate years (2020 and 2021). For this you will be using CASE WHEN statements to provide modified campaign grouping. **First, write a query to pull 3 different marketing related metrics split by campaign.**



In [ ]:
SELECT
   CASE
       WHEN Month=202011 AND Campaign="BlackFriday_V1" THEN "BlackFriday_V1_2020"
       WHEN Month=202111 AND Campaign="BlackFriday_V1" THEN "BlackFriday_V1_2021"
   ELSE Campaign END AS Campaign,
   SUM(Cost)/SUM(Impressions)*1000 AS CPM,
   SUM(Clicks)/SUM(Impressions) AS CTR,
   SUM(Cost)/SUM(Clicks) AS CPC
FROM `tc-da-1.turing_data_analytics.adsense_monthly`
GROUP BY 1;

# extra points if identified to include CTR or CPC metrics

Next you visualize your data in a [chart](https://docs.google.com/spreadsheets/d/1DKh2zCDuuaHqkm7fvWsjSFVqF1R5CJvPhyJ27Ibo3_E/edit#gid=530480965).

Your marketing manager is really happy you included additional metrics, though asks you to provide analysis and insights for them as well.

## Thursday


Thursday, having heard about your marketing campaigns analysis, the product manager comes over and tells you that website tracking data is available in a [different table](https://console.cloud.google.com/bigquery?authuser=3&project=tc-da-1&ws=!1m14!1m3!8m2!1s756497109418!2s1b8fa7cb3d73401f81fea1591fa615a7!1m4!4m3!1stc-da-1!2sturing_data_analytics!3sstackoverflow_posts!1m4!4m3!1stc-da-1!2sturing_data_analytics!3sraw_events&d=turing_data_analytics&p=tc-da-1&t=raw_events&page=table). He wonders if it would be useful for you. He lets you know that this table also contains purchases data.

You have identified 3 main metrics you want to calculate: COV% (conversion rate), CPA (cost per acquisition)  and ROI (marketing return on investment). First you have a look at the data and identify the event name which will help you identify conversions, the column which helps you identify return and the column which helps you identify users (you will need this to calculate acquired customers). You use this filtered data to join your marketing data. For simplicity reasons we will assume that marketing data is for the same period as website tracking data, thus you do not need to filter for dates at all.

**Begin by writing CTE which collects the overall purchases amount and the number of users per campaign from website tracking data. Then, write the next CTE which collects overall marketing campaign data (this time you should care only about the campaign name and do not mind that BlackFriday_V1 campaign was running for 2 different years) by campaign name. Then join both CTEs to have primary metrics which can later be used in a spreadsheet to calculate COV%, CPA and ROI. You want to have a look at primary metrics first.**




In [ ]:
WITH website_data AS (
   SELECT
       campaign,
       COUNT(1) AS purchases_count,
       COUNT(DISTINCT user_pseudo_id) AS user_count,
       SUM(purchase_revenue_in_usd) AS campaign_revenue
   FROM
       `tc-da-1.turing_data_analytics.raw_events`
   WHERE
       event_name="purchase"
   GROUP BY 1
),
marketing_data AS (
   SELECT
       Campaign as campaign,
       SUM(Clicks) AS clicks,
       SUM(Cost) AS cost
   FROM
       `tc-da-1.turing_data_analytics.adsense_monthly`
   GROUP BY 1
)
SELECT
   marketing_data.campaign,
   marketing_data.clicks,
   marketing_data.cost,
   website_data.purchases_count,
   website_data.user_count,
   website_data.campaign_revenue
FROM
   marketing_data
LEFT JOIN
   website_data
ON
   marketing_data.campaign = website_data.campaign;


You have a look at the resulting data and realize that there are no purchases captured by users coming from your marketing campaigns. You ask a product analyst to investigate this and he realizes that tracking was not correctly implemented and all marketing related purchases were registered under “(data deleted)”. You promise that tomorrow you will still come up with a good way to estimate overall marketing activities performance.


## Friday
Finally Friday, the last day before the weekend. You remember that you promised to evaluate overall marketing activities performance and calculate COV, CPA and ROI even with broken data.

Write a query which will help you evaluate the performance of marketing activities even with broken tracking. Have in mind that sometimes working as a marketing analyst you do not have perfect data, thus you need to come up with creative / good enough solutions to evaluate your marketing performance.


In [ ]:
WITH website_data AS (
   SELECT
       "All Campaigns" AS campaign,
       COUNT(1) AS purchases_count,
       COUNT(DISTINCT user_pseudo_id) AS user_count,
       SUM(purchase_revenue_in_usd) AS campaign_revenue
   FROM
       `tc-da-1.turing_data_analytics.raw_events`
   WHERE
       event_name="purchase"
       AND campaign="(data deleted)"
   GROUP BY 1
),
marketing_data AS (
   SELECT
       "All Campaigns" AS campaign,
       SUM(Clicks) AS clicks,
       SUM(Cost) AS cost
   FROM
       `tc-da-1.turing_data_analytics.adsense_monthly`
)
SELECT
   marketing_data.campaign,
   purchases_count/clicks AS COV,
   cost/user_count AS CPA,
   campaign_revenue/cost AS ROI
FROM
   marketing_data
LEFT JOIN
   website_data
ON
   marketing_data.campaign = website_data.campaign;